# **Title: Multiclass Sentiment Detection with text and speech**


**Problem statement:**A multimodal emotion detection system that accurately classifies human emotions using both textual and audio inputs by leveraging classical machine learning and deep learning models.

**Project Overview**

Goal: Build a multimodal sentiment/emotion detection system using:

Text (from sentences)
Audio (speech data)
Apply both classical ML and deep learning (DL)
Target: Classify emotions like joy, anger, sadness, surprise, fear, etc

In [ ]:
# Install Required Packages

!pip install librosa xgboost scikit-learn pandas numpy matplotlib seaborn tensorflow

In [ ]:

# Import Libraries
import os
import librosa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, GlobalAveragePooling1D, Dropout, Conv2D, MaxPooling2D, Flatten, Reshape, Input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

In [ ]:

# TEXT DATASET LOADING FROM train.txt
df_text = pd.read_csv('train.txt', sep=';', header=None, names=['text', 'emotion'])
X_text = df_text['text']
y_text = df_text['emotion']

le_text = LabelEncoder()
y_text_enc = le_text.fit_transform(y_text)

X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(X_text, y_text_enc, test_size=0.2, random_state=42)


In [ ]:
# TF-IDF for classical ML
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train_text)
X_test_vec = vectorizer.transform(X_test_text)


In [ ]:

# Classical ML on Text

text_models = {
    "NaiveBayes": MultinomialNB(),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "SVM": SVC(kernel='linear', probability=True),
    "RandomForest": RandomForestClassifier(n_estimators=200),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
}

for name, model in text_models.items():
    print(f"\n=== {name} (Text) ===")
    model.fit(X_train_vec, y_train_text)
    y_pred = model.predict(X_test_vec)
    # Convert target_names to strings to avoid TypeError
    target_names_str = [str(cls) for cls in le_text.classes_]
    print(classification_report(y_test_text, y_pred, target_names=target_names_str))


=== NaiveBayes (Text) ===
              precision    recall  f1-score   support

       anger       0.92      0.53      0.67       427
        fear       0.92      0.42      0.58       397
         joy       0.66      0.97      0.78      1021
        love       1.00      0.14      0.25       296
     sadness       0.73      0.94      0.82       946
    surprise       1.00      0.03      0.05       113

    accuracy                           0.73      3200
   macro avg       0.87      0.51      0.53      3200
weighted avg       0.79      0.73      0.68      3200


=== LogisticRegression (Text) ===
              precision    recall  f1-score   support

       anger       0.89      0.83      0.86       427
        fear       0.87      0.79      0.83       397
         joy       0.83      0.96      0.89      1021
        love       0.89      0.65      0.75       296
     sadness       0.90      0.93      0.92       946
    surprise       0.89      0.52      0.66       113

    accuracy   

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [22:34:34] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


              precision    recall  f1-score   support

       anger       0.89      0.87      0.88       427
        fear       0.85      0.87      0.86       397
         joy       0.85      0.91      0.88      1021
        love       0.80      0.81      0.80       296
     sadness       0.95      0.88      0.91       946
    surprise       0.83      0.80      0.81       113

    accuracy                           0.88      3200
   macro avg       0.86      0.85      0.86      3200
weighted avg       0.88      0.88      0.88      3200



In [ ]:
# Deep Learning (LSTM) on Text

max_words = 10000
max_len = 100
num_classes_text = len(le_text.classes_)


In [ ]:
# Tokenization for DL
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train_text)
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train_text), maxlen=max_len)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test_text), maxlen=max_len)


In [ ]:
# One-hot labels for DL
y_train_cat = to_categorical(y_train_text)
y_test_cat = to_categorical(y_test_text)


In [ ]:
# LSTM model
model_lstm = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    LSTM(64),
    Dropout(0.5),
    Dense(num_classes_text, activation='softmax')
])
model_lstm.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model_lstm.fit(X_train_seq, y_train_cat, epochs=5, batch_size=32, validation_split=0.1)


Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


360/360 ━━━━━━━━━━━━━━━━━━━━ 39s 92ms/step - accuracy: 0.3706 - loss: 1.5615 - val_accuracy: 0.7445 - val_loss: 0.8895
Epoch 2/5
360/360 ━━━━━━━━━━━━━━━━━━━━ 31s 64ms/step - accuracy: 0.8333 - loss: 0.5628 - val_accuracy: 0.8773 - val_loss: 0.3935
Epoch 3/5
360/360 ━━━━━━━━━━━━━━━━━━━━ 41s 64ms/step - accuracy: 0.9394 - loss: 0.1990 - val_accuracy: 0.8805 - val_loss: 0.3511
Epoch 4/5
360/360 ━━━━━━━━━━━━━━━━━━━━ 42s 67ms/step - accuracy: 0.9623 - loss: 0.1268 - val_accuracy: 0.9086 - val_loss: 0.3207
Epoch 5/5
360/360 ━━━━━━━━━━━━━━━━━━━━ 40s 64ms/step - accuracy: 0.9779 - loss: 0.0768 - val_accuracy: 0.9094 - val_loss: 0.3444


In [ ]:
import pickle  # Import the pickle module

In [ ]:

model_lstm.save('models/text_lstm.h5')
with open('models/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
with open('models/label_encoder_text.pkl', 'wb') as f:
    pickle.dump(le_text, f)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Load Audio Dataset (TESS)

AUDIO_PATH = '/content/drive/MyDrive/Sentiment_Audio_Dataset/TESS Toronto emotional speech set data/'
audio_files = []
audio_labels = []

for root, dirs, files in os.walk(AUDIO_PATH):
    for file in files:
        if file.endswith('.wav'):
            emotion = root.split('/')[-1].split('_')[-1].lower()
            audio_files.append(os.path.join(root, file))
            audio_labels.append(emotion)


In [ ]:
# Load Audio Dataset (TESS)

AUDIO_PATH = '/content/drive/MyDrive/Sentiment_Audio_Dataset/TESS Toronto emotional speech set data/'
audio_files = []
audio_labels = []
if not os.path.exists(AUDIO_PATH):
    raise FileNotFoundError(f"❌ Audio path '{AUDIO_PATH}' not found.")

In [ ]:
# MFCC Feature Extractor

def extract_mfcc(file_path, n_mfcc=40):
    try:
        y, sr = librosa.load(file_path, sr=22050)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        return np.mean(mfcc, axis=1)
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None


In [ ]:
# Load and Process Dataset
audio_features = []
audio_labels = []
failed_files = []

In [ ]:
# Traverse through all subfolders in the dataset
for root, dirs, files in os.walk(AUDIO_PATH):
    for file in files:
        if file.endswith('.wav'):
            # Get emotion from folder name (last part)
            folder_name = os.path.basename(root)
            emotion = folder_name.lower().split('_')[-1]  # e.g., 'YAF_angry' -> 'angry'

            file_path = os.path.join(root, file)
            mfcc = extract_mfcc(file_path)

            if mfcc is not None:
                audio_features.append(mfcc)
                audio_labels.append(emotion)
            else:
                failed_files.append(file_path)

In [ ]:
# Convert to NumPy arrays
audio_features = np.array(audio_features)
le_audio = LabelEncoder()
y_audio = le_audio.fit_transform(audio_labels)

print(f" Extracted MFCCs from {len(audio_features)} audio files.")
print(f" Skipped {len(failed_files)} corrupted or unreadable files.")


✅ Extracted MFCCs from 2800 audio files.
⚠️ Skipped 0 corrupted or unreadable files.


In [ ]:
# Train-Test Split
# ==========================
if len(audio_features) == 0:
    raise ValueError(" No valid audio samples found. Check your dataset structure.")

X_train_audio, X_test_audio, y_train_audio, y_test_audio = train_test_split(
    audio_features, y_audio, test_size=0.2, random_state=42)


In [ ]:
# ==========================
# Normalize Features
# ==========================
scaler = StandardScaler()
X_train_audio = scaler.fit_transform(X_train_audio)
X_test_audio = scaler.transform(X_test_audio)

print(f" Training samples: {X_train_audio.shape}, Testing samples: {X_test_audio.shape}")

✅ Training samples: (2240, 40), Testing samples: (560, 40)


In [ ]:
# Classical ML on Audio
# ==========================
audio_models = {
    "RandomForest": RandomForestClassifier(n_estimators=200),
    "SVM": SVC(probability=True),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
}

for name, model in audio_models.items():
    print(f"\n=== {name} (Audio) ===")
    model.fit(X_train_audio, y_train_audio)
    y_pred_audio = model.predict(X_test_audio)
    print(classification_report(y_test_audio, y_pred_audio, target_names=le_audio.classes_))



=== RandomForest (Audio) ===
              precision    recall  f1-score   support

       angry       1.00      1.00      1.00        91
     disgust       1.00      1.00      1.00        84
        fear       1.00      1.00      1.00        86
       happy       0.97      0.99      0.98        79
     neutral       1.00      1.00      1.00        80
         sad       1.00      1.00      1.00        70
    surprise       0.97      0.95      0.96        37
   surprised       1.00      1.00      1.00        33

    accuracy                           0.99       560
   macro avg       0.99      0.99      0.99       560
weighted avg       0.99      0.99      0.99       560


=== SVM (Audio) ===
              precision    recall  f1-score   support

       angry       1.00      1.00      1.00        91
     disgust       1.00      1.00      1.00        84
        fear       1.00      1.00      1.00        86
       happy       0.99      1.00      0.99        79
     neutral       1.00    

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [22:41:38] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


              precision    recall  f1-score   support

       angry       1.00      1.00      1.00        91
     disgust       1.00      0.99      0.99        84
        fear       1.00      1.00      1.00        86
       happy       0.99      1.00      0.99        79
     neutral       1.00      1.00      1.00        80
         sad       1.00      1.00      1.00        70
    surprise       1.00      0.97      0.99        37
   surprised       0.97      1.00      0.99        33

    accuracy                           1.00       560
   macro avg       0.99      1.00      0.99       560
weighted avg       1.00      1.00      1.00       560



In [ ]:
# Deep Learning CNN on Audio
# ==========================
X_train_audio_dl = np.reshape(X_train_audio, (-1, 40, 1, 1))
X_test_audio_dl = np.reshape(X_test_audio, (-1, 40, 1, 1))
y_train_audio_cat = to_categorical(y_train_audio)
y_test_audio_cat = to_categorical(y_test_audio)

model_cnn = Sequential([
    Input(shape=(40,1,1)),
    Conv2D(32, (3,1), activation='relu'),
    MaxPooling2D((2,1)),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(len(le_audio.classes_), activation='softmax')
])
model_cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model_cnn.fit(X_train_audio_dl, y_train_audio_cat, epochs=5, batch_size=32, validation_split=0.1)


Epoch 1/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.5372 - loss: 1.4266 - val_accuracy: 0.9688 - val_loss: 0.2754
Epoch 2/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9501 - loss: 0.2420 - val_accuracy: 0.9821 - val_loss: 0.1113
Epoch 3/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9811 - loss: 0.1096 - val_accuracy: 0.9821 - val_loss: 0.0717
Epoch 4/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9896 - loss: 0.0662 - val_accuracy: 0.9866 - val_loss: 0.0676
Epoch 5/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9911 - loss: 0.0553 - val_accuracy: 0.9911 - val_loss: 0.0622


In [ ]:
# Fusion Model (Text + Audio)
# ==========================

# Get the number of samples in the smaller dataset (audio training set)
num_audio_train_samples = X_train_audio.shape[0]
num_audio_test_samples = X_test_audio.shape[0]

# Select the first 'num_audio_train_samples' from the text training set
X_train_vec_subset = X_train_vec[:num_audio_train_samples].toarray()
y_train_text_subset = y_train_text[:num_audio_train_samples]

# Select the first 'num_audio_test_samples' from the text test set
X_test_vec_subset = X_test_vec[:num_audio_test_samples].toarray()
y_test_text_subset = y_test_text[:num_audio_test_samples]


# Now, the number of samples match for concatenation
X_train_fused = np.hstack((X_train_vec_subset, X_train_audio))
X_test_fused = np.hstack((X_test_vec_subset, X_test_audio))
y_fused = y_train_text_subset # Use the subset of the text labels

rf_fusion = RandomForestClassifier(n_estimators=300)
# Train the fusion model on the aligned training data
rf_fusion.fit(X_train_fused, y_fused)

# Predict on the aligned test data
y_pred_fused = rf_fusion.predict(X_test_fused)

print("\n=== Fusion Model (Text+Audio) ===")
# Evaluate using the subset of text test labels
print(classification_report(y_test_text_subset, y_pred_fused, target_names=le_text.classes_))


=== Fusion Model (Text+Audio) ===
              precision    recall  f1-score   support

       anger       1.00      0.11      0.20        72
        fear       1.00      0.15      0.25        62
         joy       0.39      0.90      0.54       168
        love       0.70      0.14      0.23        51
     sadness       0.56      0.40      0.47       191
    surprise       1.00      0.06      0.12        16

    accuracy                           0.45       560
   macro avg       0.77      0.29      0.30       560
weighted avg       0.64      0.45      0.40       560



In [ ]:
# GridSearch Example (SVM on Text)
# ==========================
params = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf']
}
gs = GridSearchCV(SVC(), params, cv=3)
gs.fit(X_train_vec, y_train_text)
print("Best SVM params:", gs.best_params_)

Best SVM params: {'C': 1, 'kernel': 'linear'}


In [ ]:
# After CNN
model_cnn.save('models/audio_cnn.h5')
with open('models/label_encoder_audio.pkl', 'wb') as f:
    pickle.dump(le_audio, f)
with open('models/audio_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [ ]:
import joblib # Import the joblib library
# After training fusion model
joblib.dump(rf_fusion, 'models/fusion_rf.pkl')
with open('models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

In [ ]:
!pip install streamlit

In [ ]:
import streamlit as st
import librosa
import numpy as np
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import joblib

# =============================
# Load Models & Tokenizers
# =============================

# Text LSTM model
model_lstm = load_model('models/text_lstm.h5')
tokenizer = pickle.load(open('models/tokenizer.pkl', 'rb'))
le_text = pickle.load(open('models/label_encoder_text.pkl', 'rb'))

# Audio CNN model
model_cnn = load_model('models/audio_cnn.h5')
le_audio = pickle.load(open('models/label_encoder_audio.pkl', 'rb'))
scaler = pickle.load(open('models/audio_scaler.pkl', 'rb'))

# Fusion Model (e.g. Random Forest)
fusion_model = joblib.load('models/fusion_rf.pkl')
vectorizer = pickle.load(open('models/tfidf_vectorizer.pkl', 'rb'))

# =============================
# Feature Extraction Functions
# =============================

def extract_audio_features(file_path):
    y, sr = librosa.load(file_path, sr=22050)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_scaled = scaler.transform([mfcc_mean])
    return mfcc_scaled, mfcc_mean.reshape(40, 1, 1)

def prepare_text(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=100)
    return padded

# =============================
# Streamlit UI
# =============================

st.title("🎭 Emotion Detection from Audio & Text")

st.sidebar.header("📎 Input Data")
text_input = st.sidebar.text_area("Enter a text message")
audio_file = st.sidebar.file_uploader("Upload a WAV audio file", type=['wav'])

if st.sidebar.button("Predict Emotion"):

    if not text_input and not audio_file:
        st.warning("Please provide either text or audio.")

    else:
        # TEXT PREDICTION
        if text_input:
            X_text_dl = prepare_text(text_input)
            pred_text = model_lstm.predict(X_text_dl)
            emotion_text = le_text.inverse_transform([np.argmax(pred_text)])[0]
            st.subheader("📝 Text Emotion Prediction")
            st.success(f"Emotion: **{emotion_text}**")

        # AUDIO PREDICTION
        if audio_file:
            with open("temp.wav", "wb") as f:
                f.write(audio_file.read())
            mfcc_scaled, mfcc_reshaped = extract_audio_features("temp.wav")
            mfcc_reshaped = mfcc_reshaped[np.newaxis, :, :, np.newaxis]
            pred_audio = model_cnn.predict(mfcc_reshaped)
            emotion_audio = le_audio.inverse_transform([np.argmax(pred_audio)])[0]
            st.subheader("🎧 Audio Emotion Prediction")
            st.success(f"Emotion: **{emotion_audio}**")

        # FUSION
        if text_input and audio_file:
            tfidf_text = vectorizer.transform([text_input]).toarray()
            fusion_input = np.hstack((tfidf_text, mfcc_scaled))
            fusion_pred = fusion_model.predict(fusion_input)
            fusion_emotion = le_text.inverse_transform(fusion_pred)[0]
            st.subheader("🔀 Fusion Model Prediction")
            st.success(f"Emotion: **{fusion_emotion}**")


In [ ]:
# ==========================
# Upload a Text + Audio File
# ==========================
from google.colab import files
uploaded = files.upload()

# Example usage:
# Provide the text manually or from a text file
input_text = "I am really happy and excited today!"

# Assuming the uploaded audio file is in uploaded.keys()
audio_file_path = list(uploaded.keys())[0]  # should be .wav file

# ==========================
# Text Preprocessing & Prediction
# ==========================
# Vectorized (ML) prediction
X_input_vec = vectorizer.transform([input_text])
pred_text_ml = text_models["RandomForest"].predict(X_input_vec)[0]
label_text_ml = le_text.inverse_transform([pred_text_ml])[0]

# LSTM (DL) prediction
X_input_seq = tokenizer.texts_to_sequences([input_text])
X_input_pad = pad_sequences(X_input_seq, maxlen=100)
pred_text_lstm = model_lstm.predict(X_input_pad)
label_text_lstm = le_text.inverse_transform([np.argmax(pred_text_lstm)])[0]

# ==========================
# Audio Preprocessing & Prediction
# ==========================
def extract_mfcc(file_path, n_mfcc=40):
    y, sr = librosa.load(file_path, sr=22050)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    return np.mean(mfcc, axis=1)

mfcc_input = extract_mfcc(audio_file_path)
mfcc_input_scaled = scaler.transform([mfcc_input])

# RandomForest (ML)
pred_audio_ml = audio_models["RandomForest"].predict(mfcc_input_scaled)[0]
label_audio_ml = le_audio.inverse_transform([pred_audio_ml])[0]

# CNN (DL)
mfcc_input_dl = np.reshape(mfcc_input_scaled, (-1, 40, 1, 1))
pred_audio_cnn = model_cnn.predict(mfcc_input_dl)
label_audio_cnn = le_audio.inverse_transform([np.argmax(pred_audio_cnn)])[0]

# ==========================
# Fusion (Text + Audio) Prediction
# ==========================
X_fused_input = np.hstack((X_input_vec.toarray(), mfcc_input_scaled))
pred_fusion = rf_fusion.predict(X_fused_input)[0]
label_fusion = le_text.inverse_transform([pred_fusion])[0]

# ==========================
# Final Results
# ==========================
print("\n TEXT INPUT:", input_text)
print(f" ML Prediction (Text): {label_text_ml}")
print(f" DL Prediction (Text - LSTM): {label_text_lstm}")

print(f"\n Audio File: {audio_file_path}")
print(f" ML Prediction (Audio): {label_audio_ml}")
print(f" DL Prediction (Audio - CNN): {label_audio_cnn}")

print(f"\n Fusion Prediction (Text + Audio): {label_fusion}")


Saving YAF_such_happy.wav to YAF_such_happy.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 231ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step

 TEXT INPUT: I am really happy and excited today!
 ML Prediction (Text): joy
 DL Prediction (Text - LSTM): joy

 Audio File: YAF_such_happy.wav
 ML Prediction (Audio): happy
 DL Prediction (Audio - CNN): happy

🔗 Fusion Prediction (Text + Audio): joy


In [ ]:
# ==========================
# Upload a Text + Audio File
# ==========================
from google.colab import files
uploaded = files.upload()

# Example usage:
# Provide the text manually or from a text file
input_text = "i don't like you"

# Assuming the uploaded audio file is in uploaded.keys()
audio_file_path = list(uploaded.keys())[0]  # should be .wav file

# ==========================
# Text Preprocessing & Prediction
# ==========================
# Vectorized (ML) prediction
X_input_vec = vectorizer.transform([input_text])
pred_text_ml = text_models["RandomForest"].predict(X_input_vec)[0]
label_text_ml = le_text.inverse_transform([pred_text_ml])[0]

# LSTM (DL) prediction
X_input_seq = tokenizer.texts_to_sequences([input_text])
X_input_pad = pad_sequences(X_input_seq, maxlen=100)
pred_text_lstm = model_lstm.predict(X_input_pad)
label_text_lstm = le_text.inverse_transform([np.argmax(pred_text_lstm)])[0]

# ==========================
# Audio Preprocessing & Prediction
# ==========================
def extract_mfcc(file_path, n_mfcc=40):
    y, sr = librosa.load(file_path, sr=22050)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    return np.mean(mfcc, axis=1)

mfcc_input = extract_mfcc(audio_file_path)
mfcc_input_scaled = scaler.transform([mfcc_input])

# RandomForest (ML)
pred_audio_ml = audio_models["RandomForest"].predict(mfcc_input_scaled)[0]
label_audio_ml = le_audio.inverse_transform([pred_audio_ml])[0]

# CNN (DL)
mfcc_input_dl = np.reshape(mfcc_input_scaled, (-1, 40, 1, 1))
pred_audio_cnn = model_cnn.predict(mfcc_input_dl)
label_audio_cnn = le_audio.inverse_transform([np.argmax(pred_audio_cnn)])[0]

# ==========================
# Fusion (Text + Audio) Prediction
# ==========================
X_fused_input = np.hstack((X_input_vec.toarray(), mfcc_input_scaled))
pred_fusion = rf_fusion.predict(X_fused_input)[0]
label_fusion = le_text.inverse_transform([pred_fusion])[0]

# ==========================
# Final Results
# ==========================
print("\n TEXT INPUT:", input_text)
print(f" ML Prediction (Text): {label_text_ml}")
print(f" DL Prediction (Text - LSTM): {label_text_lstm}")

print(f"\n Audio File: {audio_file_path}")
print(f" ML Prediction (Audio): {label_audio_ml}")
print(f" DL Prediction (Audio - CNN): {label_audio_cnn}")

print(f"\n Fusion Prediction (Text + Audio): {label_fusion}")


Saving OAF_young_ps.wav to OAF_young_ps.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step

 TEXT INPUT: i don't like you
 ML Prediction (Text): joy
 DL Prediction (Text - LSTM): anger

 Audio File: OAF_young_ps.wav
 ML Prediction (Audio): surprise
 DL Prediction (Audio - CNN): surprise

 Fusion Prediction (Text + Audio): sadness
